# Predicting Event Success for Data-Driven Marketing Decisions
### A machine learning case study for optimizing marketing investments before event publication.
This project analyzes historical data from an Italian company organizing social events and experiences. The goal is to predict whether an event will succeed or fail using only information available at publication time (T0).

## Project Overview
Marketing teams frequently invest advertising budgets without knowing whether an event is likely to succeed.
This project develops a machine learning model capable of predicting event success using only the information available at publication time (T0).
Rather than simply building a classifier, the project demonstrates how predictive analytics can support marketing planning, prioritize event evaluation and provide an early data-driven signal before launch.

## Business Problem
Launching an unsuccessful event generates unnecessary marketing costs and inefficient resource allocation.
The objective of this project is to support decision-making before publication by estimating event success potential and translating model scores into operational marketing priorities. The resulting framework is designed to support campaign prioritization, resource-allocation decisions and early event evaluation.

## Business Questions
This analysis aims to answer the following business questions:
- Which events are most likely to succeed?
- Which variables have the greatest impact on event success?
- Which cities and event categories perform best?
- Can machine learning improve marketing decisions before publication?
- How can predicted success scores support budget allocation?
- How can the model be transformed into a practical decision-support tool?

## Key Results
| KPI | Result |
|------|---------|
| Dataset | 826 historical events |
| Target | Event Success |
| Model | Logistic Regression |
| Holdout Accuracy | 70.9% |
| Holdout ROC-AUC | 0.735 |
| Holdout Average Precision | 0.304 |
| Success Recall | 68% |
| Business Goal | Early Marketing Prioritization |
| Final Output | Event Success Score & Priority Class |

## Data Privacy
The original dataset used in this project comes from internal business data. To protect company privacy and sensitive information: all personal information has been removed, the dataset has been anonymized and sensitive business details have been modified when necessary. 
The project is shared exclusively for educational and portfolio purposes. 
This repository contains only anonymized and partially modified data structures. No sensitive business information is disclosed.

## Data Preparation
Only variables available before event publication (T0) were retained.
Variables describing user behaviour after publication (such as one-week visitors and reading time evolution) were removed to simulate a realistic business scenario.
This prevents data leakage and ensures that predictions can be used operationally before launching an event.

In [ ]:
# import libraries:
%matplotlib inline
# Data manipulation
import numpy as np
import pandas as pd

# Data visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Model selection and validation
from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_validate,
    cross_val_predict
)

# Preprocessing
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Machine learning
from sklearn.linear_model import LogisticRegression

# Model evaluation
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    roc_auc_score,
    confusion_matrix,
    roc_curve,
    average_precision_score
)
# Statistical analysis
from scipy.stats import chi2_contingency

# Notebook utilities
from IPython.core.display import HTML as Center

# Warnings
from warnings import filterwarnings
filterwarnings("ignore")

In [ ]:
#Centering the plots
Center(""" <style>
.jp-RenderedImage {
    display: table-cell;
    text-align: center;
    vertical-align: middle;
}
</style> """)

In [ ]:
# Seaborn visualization settings:
sns.set_theme(
    style="darkgrid",      
    context="notebook",        
    palette="deep"
)

In [ ]:
plt.style.use('bmh')

plt.rcParams['figure.figsize'] = (7,4)
plt.rcParams['axes.titlesize'] = 16
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['xtick.labelsize'] = 10
plt.rcParams['ytick.labelsize'] = 10
plt.rcParams['legend.fontsize'] = 10
plt.rcParams['figure.dpi'] = 120

In [ ]:
# Load cleaned dataset
df = pd.read_excel("anonimized_data.xlsx")

# Remove variables unavailable at publication time
df_t0 = df.drop(columns=[
    "VISUALIZATIONS + 1 WEEK",
    "DELTA VISUALIZATIONS",
    "READING TIME",
    "READING TIME(s)",
    "READING TIME +1 WEEK",
    "READING TIME +1 WEEK(s)"
])

df_t0.head()

> **Target Variable**
>
> **Success**
> - **1** → Successful event
> - **0** → Unsuccessful event

The cleaned dataset contains only information available at the time of publication (T0), ensuring that the machine learning model predicts event success without using future information and avoiding data leakage.

In [ ]:
# Check missing values
print(df_t0.isnull().sum()) 

In [ ]:
# Dataset structure and variable types
print(df_t0.info())

## Exploratory Data Analysis
### Target Distribution

In [ ]:
target_distribution = (
    df_t0["STATE"]
    .value_counts(normalize=True)
    .rename_axis("Outcome")
    .reset_index(name="Percentage")
)

display(target_distribution)

> **Business Insight:** 
>Approximately 87% of historical events were unsuccessful.
>This strong imbalance reflects a realistic business scenario where successful events are relatively rare, making minority-class identification more challenging.

### Top Performing Cities

In [ ]:
# Analyze historical success rate by city 
city_analysis = df_t0.groupby('CITY')['STATE'].agg(['count', 'sum'])

city_analysis['success_rate'] = city_analysis['sum'] / city_analysis['count']

city_analysis = city_analysis.sort_values(by='success_rate', ascending=False)
city_analysis = city_analysis[city_analysis['count'] > 5]
city_analysis.head(10)

In [ ]:
# Top performing cities
plt.figure(figsize=(10,5))

top_cities = city_analysis.sort_values(
    by="success_rate",
    ascending=False
).head(10)

bars = plt.bar(
    top_cities.index,
    top_cities["success_rate"],
    edgecolor="black"
)

for bar in bars:
    plt.text(
        bar.get_x() + bar.get_width()/2,
        bar.get_height() + 0.015,
        f"{bar.get_height():.2f}",
        ha="center",
        fontsize=10,
        fontweight="bold"
    )

plt.title("Top 10 Cities by Historical Success Rate", fontsize=15, fontweight="bold")
plt.xlabel("City")
plt.ylabel("Historical Success Rate")
plt.ylim(0,1)
plt.xticks(rotation=45)
sns.despine()
plt.tight_layout()
plt.savefig(
    "Top_10_cities.png",
    dpi=300,
    facecolor="white",
    bbox_inches="tight"
)
plt.show()

> **Business Insight:**
Historical performance varies considerably across cities. Some locations show higher historical success rates within the observed dataset, suggesting that geographical context plays a significant role in event performance.

### Event Categories

In [ ]:
#  Best performing events
categoria_analysis = df_t0.groupby('EVENT CATEGORY')['STATE'].agg(['count', 'sum'])

categoria_analysis['success_rate'] = categoria_analysis['sum'] / categoria_analysis['count']
categoria_analysis = categoria_analysis[categoria_analysis['count'] >= 10]
categoria_analysis.sort_values(by='success_rate', ascending=False)

>  **Business Insight:**
> Event performance varies considerably across categories. **Category_4** achieved the highest historical success rate (17.28%) while also representing the largest share of events, suggesting that it it represents a historically strong and operationally relevant event category. Conversely, **Category_1** recorded no successful events during the observed period, suggesting that this category may require further evaluation before additional marketing resources are allocated.

### Days of the week performance

In [ ]:
# Best performing days
day_analysis = df_t0.groupby('DAY')['STATE'].agg(['count', 'sum'])

day_analysis['success_rate'] = day_analysis['sum'] / day_analysis['count']

day_analysis.sort_values(by='success_rate', ascending=False)

>  **Business Insight**
> Event performance varies across the days of the week. **Day_4** achieved the highest historical success rate (15.13%), followed closely by **Day_5** and **Day_7**. In contrast, **Day_1** showed the lowest success rate (3.92%), suggesting that scheduling events on this day may require additional marketing efforts or a different promotional strategy.  However, these descriptive differences should be interpreted cautiously. Statistical validation is required to determine whether the observed variation reflects a systematic relationship with event success.

> ***Key Insights:*** <br>
> Main findings from the exploratory analysis:
>- Event success strongly varies across cities
>- Some event categories consistently outperform others
>- Several categories show systematic underperformance
>- Temporal patterns appear less pronounced than geographical differences

## Statistical validation
To validate the patterns observed during exploratory analysis, a Chi-Square test of independence was applied to the main categorical variables.
The test evaluates whether each feature is statistically associated with event success, while Cramér's V measures the strength of the observed association.

In [ ]:
# Evaluate statistical association with event success
def categorical_association(data, feature, target):

    contingency = pd.crosstab(data[feature], data[target])

    chi2, p_value, _, _ = chi2_contingency(contingency)

    n = contingency.to_numpy().sum()

    cramers_v = np.sqrt(
        chi2 / (n * (min(contingency.shape) - 1))
    )

    return chi2, p_value, cramers_v


# Categorical variables to evaluate
statistical_features = {
    "Event Category": "EVENT CATEGORY",
    "City": "CITY",
    "Day of the Week": "DAY",
    "Month": "MONTH",
    "Target Age Group": "AGE GROUP"
}


# Run statistical tests
statistical_results = []

for label, feature in statistical_features.items():

    chi2, p_value, cramers_v = categorical_association(
        df_t0,
        feature,
        "STATE"
    )

    statistical_results.append({
        "Variable": label,
        "Chi²": chi2,
        "p-value": p_value,
        "Cramer's V": cramers_v
    })


statistical_results = pd.DataFrame(statistical_results)

display(
    statistical_results.style
        .hide(axis="index")
        .format({
            "Chi²": "{:.2f}",
            "p-value": "{:.4f}",
            "Cramer's V": "{:.3f}"
        })
)

| Variable | EDA Observation | Statistical Evidence | Analytical Interpretation |
|:---------|:----------------|:---------------------|:--------------------------|
| City | Large differences observed | ✅ Significant | Strongest observed association |
| Event Category | Large differences observed | ✅ Significant | Meaningful association |
| Target Age Group |   Differences observed   | ✅ Significant | Weak association | Statistically significant but weak association |
| Month | Apparent differences | ❌ Not significant | Limited standalone value |
| Day of the Week | Small differences | ❌ Not significant | Limited standalone value |

The Chi-Square tests identified **Event Category**, **City** and **Target Age Group** as statistically significant factors associated with event success (*p* < 0.05). Among the variable analyzed, City showed the strongest association with the target (Cramér's V = 0.320), while Event Category showed a weaker but statistically significant association (Cramér's V = 0.220). Target Age Group was also statistically significant, although the relatively low Cramér's V value (0.116) indicates that the strength of the association is weak.
In contrast, Day of the Week and Month did not show statistically significant associations with event success, suggesting limited standalone explanatory power.

> 💼 **Business Insight** The statistical analysis identified Event Category, City and Target Age Group as variables showing a significant association with event success (p < 0.05).City showed the strongest observed association, while the relationship with Target Age Group was comparatively weak.  In contrast, day of the week and month did not exhibit statistically significant relationships with the target variable.
Overall, the statistical validation reinforces the business intuition that **where** an event takes place and **what type of event** it is, are more influential than **when** it is scheduled.

## Feature Selection
The predictive model uses only variables available at event publication time (T0).
The selected features represent the event's geographical, structural, temporal and demographic characteristics:
- City
- Event Category
- Day of the Week
- Month
- Target Age Group <br>
Post-publication engagement variables were intentionally excluded to preserve the operational validity of the T0 prediction framework and prevent future-information leakage. 

In [ ]:
# Define features and target
features = [
    'CITY',
    'EVENT CATEGORY',
    'DAY',
    'MONTH',
    'AGE GROUP',
]

X = df_t0[features]
y = df_t0['STATE']
# All predictors available at T0 are categorical variables
categorical_features = features

In [ ]:
# Split the dataset while preserving the original class distribution
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y, #This preserves the original success/failure distribution in both subsets, which is particularly important given the strong target-class imbalance.
    random_state=42
)

print("Training set:", X_train.shape)
print("Test set:", X_test.shape)

print("\nTraining target distribution:")
print(y_train.value_counts(normalize=True))

print("\nTest target distribution:")
print(y_test.value_counts(normalize=True))

#### Preprocessing Pipeline
Categorical variables are encoded using a preprocessing pipeline based on `ColumnTransformer` and `OneHotEncoder`.
The encoder is fitted exclusively on the training data and automatically applies the fitted transformation to new event data.
Unknown categories are handled using `handle_unknown="ignore"`, ensuring that the prediction workflow remains robust when new cities or event categories appear.
Combining preprocessing and model training within a single pipeline also prevents inconsistencies between training and inference.

In [ ]:
# Define categorical preprocessing
categorical_preprocessor = OneHotEncoder(
    handle_unknown="ignore",
    drop="first"
)

# Apply preprocessing to categorical features
preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            categorical_preprocessor,
            categorical_features
        )
    ]
)

## Machine Learning Model
A Logistic Regression classifier was selected because it provides transparent and interpretable predictions.
SMOTE was tested to handle class imbalance. However, results did not improve significantly. In particular, the recall of the success class decreased by approximately 11%, indicating a lower ability of the model to correctly identify successful events. The final model therefore uses: `class_weight="balanced"` to increase the relative importance of minority-class observations during model training.


In [ ]:
# Build the complete machine learning pipeline
model = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),
        (
            "classifier",
            LogisticRegression(
                max_iter=1000,
                class_weight="balanced",
                random_state=42
            )
        )
    ]
)

# Train the complete pipeline
model.fit(X_train, y_train)

#### Cross-Validation
Because successful events represent a minority class, model performance may vary depending on the specific train-test split.
To evaluate model stability, a 5-fold Stratified Cross-Validation strategy was applied.
Stratification preserves the original class distribution within each fold, providing a more reliable estimate of the model's generalization performance.

In [ ]:
# Define stratified cross-validation
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Metrics selected according to the business objective
scoring = {
    "roc_auc": "roc_auc",
    "recall_success": "recall",
    "precision_success": "precision",
    "average_precision": "average_precision"
}

# Evaluate the complete pipeline
cv_results = cross_validate(
    model,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=-1
)

In [ ]:
# Summarize cross-validation performance
cv_summary = pd.DataFrame({
    "Metric": [
        "ROC-AUC",
        "Success Recall",
        "Success Precision",
        "Average Precision"
    ],
    "Mean CV Score": [
        cv_results["test_roc_auc"].mean(),
        cv_results["test_recall_success"].mean(),
        cv_results["test_precision_success"].mean(),
        cv_results["test_average_precision"].mean()
    ],
    "Standard Deviation": [
        cv_results["test_roc_auc"].std(),
        cv_results["test_recall_success"].std(),
        cv_results["test_precision_success"].std(),
        cv_results["test_average_precision"].std()
    ]
})

display(
    cv_summary.style
        .hide(axis="index")
        .format({
            "Mean CV Score": "{:.3f}",
            "Standard Deviation": "{:.3f}"
        })
)

Stratified 5-fold cross-validation produced a mean ROC-AUC of 0.654 (± 0.096) and an Average Precision of 0.243 (± 0.060).
Success-class recall averaged 54.5%, with greater variability across folds (± 15.4 percentage points). This variability reflects the difficulty of identifying a relatively small minority of successful events using only static information available before publication.
The cross-validation results suggest that T0 event characteristics provide a moderate early predictive signal, but prediction uncertainty remains substantial across different data partitions.
From a business perspective, these findings reinforce the role of the T0 model as an initial screening tool rather than a definitive success classifier. Predictions can be updated after publication when early engagement signals become available in the second stage of the decision framework.

## Model evaluation

In [ ]:
# Generate predictions on the holdout test set
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

# Calculate evaluation metrics
test_accuracy = accuracy_score(y_test, y_pred)
test_roc_auc = roc_auc_score(y_test, y_prob)
test_average_precision = average_precision_score(y_test, y_prob)

print("=== CLASSIFICATION REPORT ===")
print(classification_report(y_test, y_pred))

print(f"Accuracy: {test_accuracy:.3f}")
print(f"ROC-AUC: {test_roc_auc:.3f}")
print(f"Average Precision: {test_average_precision:.3f}")

The final holdout test evaluation produced an **accuracy of 70.9%**, a **ROC-AUC of 73.5%** and an **Average Precision of 0.304**.
The model correctly identified approximately **68% of successful events**, while success-class precision remained at **27%**. In contrast, unsuccessful events were identified with **94% precision**.
The holdout test produced stronger predictive performance than the average observed during cross-validation. For this reason, model interpretation considers both the holdout evaluation and the cross-validation results rather than relying on a single train-test split.
Overall, the model demonstrates a moderate ability to rank potentially successful events, while the variability observed across cross-validation folds highlights the uncertainty associated with predictions based exclusively on pre-publication characteristics.
Given the strong class imbalance and the limited predictive information available at T0, the model is positioned as an early-stage screening and prioritization tool rather than a final decision system.



Additional historical performance features were evaluated, including city success rate, category success rate and combined city-category success rate.
Although these variables increased model complexity, they did not produce meaningful improvements in predictive performance, suggesting that event success depends on multiple interacting factors beyond historical averages.

In [ ]:
fpr, tpr, thresholds = roc_curve(
    y_test,
    y_prob
)

plt.figure(figsize=(6, 6))

plt.plot( fpr,tpr, linewidth=2,label=f"AUC = {test_roc_auc:.2f}"
)

plt.plot( [0, 1], [0, 1], linestyle="--", alpha=0.7
)

plt.title("ROC Curve - Logistic Regression",weight="bold", pad=12
)

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")

plt.legend(loc="lower right")
plt.grid(alpha=0.3)

sns.despine()
plt.tight_layout()

plt.show()

The holdout **ROC-AUC of 0.735** indicates moderate discriminative ability. The model performs better than random discrimination and shows an ability to rank potentially successful events, although cross-validation results indicate that predictive performance varies across data partitions.


In [ ]:
cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

print(f"True Negatives: {tn}")
print(f"False Positives: {fp}")
print(f"False Negatives: {fn}")
print(f"True Positives: {tp}")

sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False)
plt.title( "Confusion Matrix - Logistic Regression",
    weight='bold',
    pad=12
)
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.savefig("conf_matrix.png",dpi=300,facecolor="white", bbox_inches="tight"
)
plt.show()

The confusion matrix provides a detailed view of classification errors on the holdout test set.
The model correctly classified 102 unsuccessful events (True Negatives) and 15 successful events (True Positives). It identified 15 out of 22 successful events, corresponding to a success-class recall of approximately 68%, while 7 successful events were missed (False Negatives).
The model also generated 41 False Positives, classifying some unsuccessful events as potentially successful. This contributes to the relatively low success-class precision of 27%.

> 💼 **Business Insight**
> From a business perspective, this error profile is consistent with the model's role as an early-stage screening tool. False positives increase the number of events requiring further evaluation, while false negatives represent potentially successful events that may be overlooked. The model therefore provides an initial prioritization signal, but predicted high-potential events should undergo additional evaluation before marketing resources are allocated

#### Logistic Regression coefficients

In [ ]:
# Extract fitted preprocessing and classifier components
fitted_preprocessor = model.named_steps["preprocessor"]
fitted_classifier = model.named_steps["classifier"]

# Retrieve encoded feature names
feature_names = (
    fitted_preprocessor
    .get_feature_names_out()
)

# Extract Logistic Regression coefficients
coeff = pd.Series(
    fitted_classifier.coef_[0],
    index=feature_names
)

# Clean feature names for visualization
coeff.index = (
    coeff.index
    .str.replace(
        "categorical__",
        "",
        regex=False
    )
)

# Inspect extracted coefficients
coeff.sort_values()

In [ ]:
# Select the strongest positive and negative coefficients
top_coefficients = (coeff.sort_values().tail(8))

bottom_coefficients = (coeff.sort_values().head(8))

selected_coefficients = pd.concat([bottom_coefficients,top_coefficients])

In [ ]:
plt.figure(figsize=(10, 6))

selected_coefficients.sort_values().plot( kind="barh")

plt.title( "Strongest Logistic Regression Coefficients",weight="bold", pad=12)

plt.xlabel("Logistic Regression Coefficient")

sns.despine()
plt.tight_layout()

plt.savefig( "features.png",dpi=300,facecolor="white",bbox_inches="tight")

plt.show()

Logistic Regression coefficients were analyzed to identify the event characteristics most strongly associated with higher or lower predicted success scores. Positive coefficients indicate categories associated with higher log-odds of predicted event success, while negative coefficients indicate categories associated with lower predicted success, holding the other model variables constant.
Because categorical predictors are represented through one-hot encoding, each coefficient is interpreted relative to the corresponding reference category excluded during encoding. The strongest positive and negative coefficients include specific cities, event categories and demographic segments, suggesting that geographical context, event format and target audience contribute to the model's prediction.
These coefficients represent predictive associations rather than causal effects and should therefore be interpreted as model signals rather than determinants of event performance.

### Business-Oriented Probability Segmentation
The model produces a continuous probability score representing the estimated likelihood of event success.
To translate these scores into operational marketing priorities, **out-of-fold predicted probabilities were generated for the training data using stratified cross-validation**.
Each training event is therefore scored by a model that was not fitted on that specific observation, providing a more realistic representation of out-of-sample model behaviour.
The average predicted probability observed among unsuccessful and successful events was then used as an empirical reference for defining three operational priority bands.
These thresholds are not interpreted as strict success boundaries. Instead, they provide a practical segmentation framework for translating continuous model scores into marketing priority levels.
The holdout test set remains excluded from threshold definition and is used exclusively for final model evaluation.

In [ ]:
# Generate out-of-fold probabilities for operational segmentation
oof_probabilities = cross_val_predict(
    model,
    X_train,
    y_train,
    cv=cv,
    method="predict_proba",
    n_jobs=-1
)[:, 1]

In [ ]:
# Compare out-of-fold probabilities by actual event outcome
probability_analysis = pd.DataFrame({
    "Actual Outcome": y_train.to_numpy(),
    "Predicted Probability": oof_probabilities
})

probability_summary = (
    probability_analysis
    .groupby("Actual Outcome")["Predicted Probability"]
    .agg([
        "count",
        "mean",
        "median",
        "std"
    ])
)

display(
    probability_summary.style
        .format({
            "mean": "{:.3f}",
            "median": "{:.3f}",
            "std": "{:.3f}"
        })
)

The out-of-fold probability analysis shows that historically successful events received a higher average predicted score (0.506) than unsuccessful events (0.392). However, the relatively similar standard deviations observed across both groups (0.218 and 0.214) indicate substantial variability in model scores and suggest considerable overlap between the two outcome distributions.
This pattern is consistent with the cross-validation results: pre-publication event characteristics provide a moderate early predictive signal but do not produce a clear separation between successful and unsuccessful events.

In [ ]:
plt.figure(figsize=(8, 5))

sns.histplot(
    data=probability_analysis,
    x="Predicted Probability",
    hue="Actual Outcome",
    bins=20,
    kde=True,
    element="step",
    stat="density",
    common_norm=False
)

plt.title(
    "Out-of-Fold Score Distribution by Actual Outcome", weight="bold", pad=12)

plt.xlabel("Predicted Success Score")
plt.ylabel("Density")

sns.despine()
plt.tight_layout()

plt.show()

The out-of-fold score distributions show a moderate shift toward higher predicted scores for historically successful events. Successful events received a mean score of 0.506, compared with 0.392 for unsuccessful events.
However, substantial overlap remains between the two distributions, indicating that pre-publication event characteristics alone do not provide a clear separation between successful and unsuccessful outcomes.

In [ ]:
# Calculate outcome-specific mean predicted probabilities
failure_mean_probability = (
    probability_analysis
    .loc[
        probability_analysis["Actual Outcome"] == 0,
        "Predicted Probability"
    ]
    .mean()
)

success_mean_probability = (
    probability_analysis
    .loc[
        probability_analysis["Actual Outcome"] == 1,
        "Predicted Probability"
    ]
    .mean()
)

medium_threshold = failure_mean_probability
high_threshold = success_mean_probability

print(
    f"Low / Medium threshold: "
    f"{medium_threshold:.3f}"
)

print(
    f"Medium / High threshold: "
    f"{high_threshold:.3f}"
)

> 💼 **Business Insight**
>
> The out-of-fold probability analysis shows that the model assigns different average success scores to historically unsuccessful and successful events. These outcome-specific mean predicted scores are used as empirical reference points for operational segmentation:
>
> * **Low Potential:** scores below the average observed among unsuccessful events
> * **Medium Potential:** scores between the two outcome-specific averages
> * **High Potential:** scores at or above the average observed among successful events
> The thresholds are not intended to represent strict success boundaries. Instead, they provide an interpretable framework for converting continuous model scores into operational marketing priority levels.


## Marketing Decision Support System
The trained model is used to simulate the probability of success for new hypothetical events. This transforms the model from a pure prediction tool into a decision-support system for event planning.

In [ ]:
# Simulate hypothetical future events
new_events = pd.DataFrame({
    "CITY": ['Citta_1','Citta_56','Citta_4','Citta_60'],
    "EVENT CATEGORY": ['categoria_8','categoria_9','categoria_1', 'categoria_5', ],
    "DAY": ['Giorno_5', 'Giorno_4','Giorno_3','Giorno_5'],
    "MONTH": ['Mese_1', 'Mese_10', 'Mese_5','Mese_8'],
    "AGE GROUP": ['Fascia_2', 'Fascia_1','Fascia_2', 'Fascia_1'],
})

In [ ]:
# Generate predicted scores using the complete pipeline
new_events["success_score"] = (
    model.predict_proba(
        new_events[features]
    )[:, 1]
)

In [ ]:
# Convert probabilities into business-oriented priority classes
def classify_event(score):

    if score >= high_threshold:
        return "🟢 HIGH POTENTIAL"

    elif score >=  medium_threshold:
        return "🟡 MEDIUM POTENTIAL"

    else:
        return "🔴 LOW POTENTIAL"


new_events["priority"] = (new_events["success_score"].apply(classify_event))

new_events

In [ ]:
plt.figure(figsize=(6,4))

plt.bar(
    new_events["CITY"],
    new_events["success_score"]
)

plt.axhline(high_threshold,linestyle="--",linewidth=1.8,label=(
        f"Successful Events Mean "
        f"({high_threshold:.2f})"
    )
)
plt.axhline(medium_threshold,linestyle="--",linewidth=1.2,label=(
        f"Unsuccessful Events Mean "
        f"({medium_threshold:.2f})"
    )
)

plt.title( "Predicted Success Score for New Events", weight='bold', pad=12)
plt.ylabel("Predicted Success Score" )
plt.xlabel("City")
plt.xticks(rotation=45)
plt.ylim(0,1)
plt.legend()
sns.despine()
plt.tight_layout()
plt.show()

# Business Conclusions

> ***Which events are most likely to succeed?*** Historical analysis revealed substantial differences in event outcomes across cities, event categories and target age groups. Statistical validation identified City and Event Category as the strongest standalone categorical associations with event success, while Target Age Group showed a weaker but statistically significant relationship. <br>
> ***Which variables have the greatest impact on event success?*** Logistic Regression coefficient analysis identified specific cities, event categories and demographic segments among the strongest positive and negative model signals. These coefficients represent predictive associations rather than causal effects and are interpreted relative to the corresponding reference categories. <br>
>***Which cities and event categories perform best?***  Historical analysis revealed substantial differences across both cities and event categories. Some locations and categories consistently outperform others, suggesting that marketing resources should be concentrated on combinations with historically stronger performance. <br>
>***Can machine learning improve marketing decisions before publication?*** Yes, although the predictive signal available at T0 remains moderate. The final holdout evaluation achieved an accuracy of 70.9%, a ROC-AUC of 0.735 and an Average Precision of 0.304. Cross-validation results showed lower average performance and variability across data partitions, highlighting the uncertainty associated with predictions based exclusively on pre-publication characteristics.<br>
While predictive performance is moderate due to the highly imbalanced dataset and the limited information available before publication, the model provides a reliable first screening of event potential and supports more informed marketing decisions.<br>
> ***How can predicted success scores support budget allocation?*** Rather than treating every event equally, model scores can be translated into Low, Medium and High Potential priority bands. High Potential events can be prioritized for additional marketing evaluation and considered for increased promotional investment, while Medium and Low Potential events may require further review, optimization or monitoring before additional resources are allocated. <br>
> ***How can the model be transformed into a practical decision-support tool?*** The complete preprocessing and Logistic Regression pipeline can directly evaluate new event data and generate a predicted success score. These continuous scores are translated into three operational priority classes:
>- 🟢 High Potential
>- 🟡 Medium Potential
>- 🔴 Low Potential <br>
> The resulting framework provides an interpretable early-stage screening system designed to support, rather than replace, managerial decision-making.

## Key Takeaway

This project demonstrates how machine learning can support marketing decision-making before an event is published.
Rather than replacing human decision-making, the proposed framework augments managerial judgment by providing an objective, data-driven success score based exclusively on information available at publication time.
The model transforms historical event data into an operational screening framework that can help marketing teams prioritize event evaluation and support more structured resource-allocation decisions. Although the predictive signal available at T0 remains moderate and varies across data partitions, the project illustrates how machine learning outputs can be translated into interpretable business priority levels and integrated into a practical decision-support workflow.

## Stage 2 – Post-Publication Prediction (T+1)

This notebook represents the first stage of the decision-making framework (**T0 Prediction**), where success is estimated before publication using only static event information.
The second stage (**T+1 Prediction**) incorporates user engagement metrics collected during the first week after publication, allowing marketing managers to update predictions and optimize advertising investments as new information becomes available.
Together, the T0 and T+1 models are designed as a two-stage decision-support framework covering the early lifecycle of an event.